# OpenDC Demo 7: Checkpointing

In the previous [demo](6.failures.ipynb), we explored the significant impact failures can have on the operations of datacenters. In this demo, we look at checkpointing, a popular technique to reduce the impact of failures. 

One problem with failures is that when a task fails, it needs to rerun all the work it had already done. When using checkpointing, tasks periodically make a snapshot. When a failure happens, the task can restart from this snapshot instead of fully starting over. 

However, checkpointing is not free. Taking a snapshot causes overhead. If no failures occur, this is wasted time. Checkpointing is thus a trade-off between the time spent making snapshots and the time saved by not backtracking. 

In this demo, we run different checkpointing models to see how they impact the performance of a system. 

The experiments shown here are inspired by the [Cloud Uptime Paper](https://arxiv.org/abs/2504.09476).

## Experiment

To use checkpointing, users have to enable it in the experiment file. [experiments/7.checkpointing/checkpointing_experiment.json](experiments/7.checkpointing/checkpointing_experiment.json) shows an experiment that uses failures. 

A Checkpointing model is provided using the `checkpointModels` parameter:

``` json
"checkpointModels": [
    null,
    {
        "interval": "10 min",
        "duration": "2 min",
        "intervalScaling": 1
    }
],
```

In this example, we are using two checkpointing models. The first, `null`, means that no checkpointing is used. This allows us to run the scenarios with and without checkpointing from one experiment file. 

The second element defines a checkpointing model using three parameters. `interval` and `duration` define how long a task waits before it makes a snapshot and how long it takes to make a snapshot, respectively. `intervalScaling` can be used to change the interval time between snapshots during the task execution. For example, when the `intervalScaling` is 1.5, the interval time is multiplied by 1.5 after each snapshot. 

**Exercise 1:** 
Extend the experiment file located [here](experiments/7.checkpointing/checkpointing_experiment.json) with more checkpointing models by changing the `interval` and `intervalScaling`. We suggest trying a wide range of values. You should also include at least one adaptive model that uses `intervalScaling`.

<details>
<summary>Click to reveal the answer to Exercise 1</summary>

See the [answer experiment file](experiments/7.checkpointing_answers/checkpointing_experiment.json).

In the answers, we added four more checkpointing models: three with a longer fixed interval (1 hour, 10 hours, and 24 hours), and one adaptive model that starts with an interval of 10 minutes and multiplies it by 1.5 after each snapshot. All models take 2 minutes to make a snapshot. We will use these models for the subsequent answers as well.

OpenDC numbers the scenarios in the order of the `checkpointModels` list. In the answer, scenario 0 uses no checkpointing, scenario 1 checkpoints every 10 minutes, scenario 2 every hour, scenario 3 every 10 hours, scenario 4 every 24 hours, and scenario 5 uses the adaptive model.

</details>

# Running an Experiment

An experiment can be run directly from the terminal using the OpenDCExperimentRunner.

In [ ]:
import subprocess

pathToExperiment = "experiments/7.checkpointing/checkpointing_experiment.json"
result = subprocess.run(["OpenDCExperimentRunner/bin/opendc", "run", pathToExperiment])

**Note:** 
Because of the failures, not all tasks can be completed. 
When a task fails too many times, it is terminated and removed from the system. This will produce a warning message.

# Output

**Exercise 2:** 
Load the results into Pandas DataFrames.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Put your code here

<details>
<summary>Click to reveal the answer to Exercise 2</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

checkpointing_models = ["no", "10min", "1h", "10h", "24h", "adaptive"]

df_hosts = {}
df_powerSources = {}
df_tasks = {}
df_services = {}

for i, model in enumerate(checkpointing_models):
    df_hosts[model] = pd.read_parquet(f"output/7.checkpointing/raw-output/{i}/seed=0/host.parquet")
    df_powerSources[model] = pd.read_parquet(f"output/7.checkpointing/raw-output/{i}/seed=0/powerSource.parquet")
    df_tasks[model] = pd.read_parquet(f"output/7.checkpointing/raw-output/{i}/seed=0/task.parquet")
    df_services[model] = pd.read_parquet(f"output/7.checkpointing/raw-output/{i}/seed=0/service.parquet")
```

</details>

# Aggregated metrics

To understand the impact of checkpointing, we can look at some aggregated metrics. 

For this purpose, we suggest using the following metrics:
1. runtime
2. energy_usage
3. carbon_emission

You can also look at some failure-specific metrics, such as:
1. tasks_completed
2. tasks_terminated

**Exercise 3:**
Explore the impact of checkpointing using aggregated metrics.

In [ ]:
# Your code goes here...

<details>
<summary>Click to reveal the answer to Exercise 3</summary>

```python
# runtime
for model in checkpointing_models:
    runtime = pd.to_timedelta(df_services[model].timestamp.max(), unit="ms")
    print(f"The workload took {runtime} with {model} checkpointing")
print()

# energy_usage
for model in checkpointing_models:
    energy = df_powerSources[model].energy_usage.max() / 3_600_000 # convert from J to kWh
    print(f"The workload consumed {energy :.2f} kWh with {model} checkpointing")
print()

# carbon_emissions
for model in checkpointing_models:
    carbon = df_powerSources[model].carbon_emission.max() / 1_000 # convert from g to kg
    print(f"The workload emitted {carbon :.2f} kg of CO2 with {model} checkpointing")
print()

# tasks_completed
for model in checkpointing_models:
    completed = df_services[model].tasks_completed.max()
    print(f"{completed} tasks were completed with {model} checkpointing")
print()

# tasks_terminated
for model in checkpointing_models:
    terminated = df_services[model].tasks_terminated.max()
    print(f"{terminated} tasks were terminated with {model} checkpointing")
```

</details>

# Service Level Agreement (SLA) violation analysis

In [demo 3](3.horizontal_scaling.ipynb) and [demo 6](6.failures.ipynb), we determined the impact of our techniques on Service Level Agreements (SLAs). 

Let's do the same for datacenters that use checkpointing. 

**Exercise 4:** 
Determine the impact of checkpointing on the delays and SLAs.

**Note:** in [demo 6](6.failures.ipynb), we used the `scheduling_delay` and `failure_delay`. However, now that we are using checkpointing, we also want to use `checkpoint_delay`. This is the delay caused by creating snapshots.

Looking at the `checkpoint_delay` vs. the other delays will provide insight into how and why a checkpointing model works.

In [ ]:
# Your code goes here

<details>
<summary>Click to reveal the answer to Exercise 4</summary>

```python
scheduling_delays = {}
failure_delays = {}
checkpoint_delays = {}
total_delays = {}

mean_scheduling_delays = {}
mean_failure_delays = {}
mean_checkpoint_delays = {}
mean_total_delays = {}

for model in checkpointing_models:
    df_tasks[model]["total_delay"] = df_tasks[model]["scheduling_delay"] + df_tasks[model]["failure_delay"] + df_tasks[model]["checkpoint_delay"]

    scheduling_delays[model] = df_tasks[model].groupby('task_id')['scheduling_delay'].agg('max') / 60_000
    failure_delays[model] = df_tasks[model].groupby('task_id')['failure_delay'].agg('max') / 60_000
    checkpoint_delays[model] = df_tasks[model].groupby('task_id')['checkpoint_delay'].agg('max') / 60_000
    total_delays[model] = df_tasks[model].groupby('task_id')['total_delay'].agg('max') / 60_000

    mean_scheduling_delays[model] = scheduling_delays[model].mean()
    mean_failure_delays[model] = failure_delays[model].mean()
    mean_checkpoint_delays[model] = checkpoint_delays[model].mean()
    mean_total_delays[model] = total_delays[model].mean()

# scheduling delay
for model in checkpointing_models:
    print(f"The average scheduling delay with {model} checkpointing is: {scheduling_delays[model].mean():.2f} minutes")

print()
# failure delay
for model in checkpointing_models:
    print(f"The average failure delay with {model} checkpointing is: {failure_delays[model].mean():.2f} minutes")

print()
# checkpoint delay
for model in checkpointing_models:
    print(f"The average checkpoint delay with {model} checkpointing is: {checkpoint_delays[model].mean():.2f} minutes")

print()
# total delay
for model in checkpointing_models:
    print(f"The average total delay with {model} checkpointing is: {total_delays[model].mean():.2f} minutes")

df_delays = pd.DataFrame({
    "Scheduling delay": mean_scheduling_delays,
    "Failure delay": mean_failure_delays,
    "Checkpoint delay": mean_checkpoint_delays,
})

ax = df_delays.plot.bar(stacked=True, color=["#2a78d6", "#eb6834", "#1baf7a"],
                        edgecolor="white", rot=0)

# Total delay on top of each bar
for x, model in enumerate(checkpointing_models):
    ax.text(x, mean_total_delays[model], f"{mean_total_delays[model]:.0f}", ha="center", va="bottom")

ax.set_ylim(top=max(mean_total_delays.values()) * 1.1)  # headroom for the labels
plt.title("Mean task delay per checkpointing model")
plt.xlabel("Checkpointing model")
plt.ylabel("Mean delay (minutes)")
plt.show()

# SLA violations
SLA = 120

SLA_violations = {}
SLA_percentages = {}
for model in checkpointing_models:
    SLA_violations[model] = (total_delays[model] > SLA).sum()
    SLA_percentages[model] = SLA_violations[model] / len(total_delays[model]) * 100

for model in checkpointing_models:
    print(f"The number of SLA violations with {model} checkpointing is: {SLA_violations[model]}")

print()
for model in checkpointing_models:
    print(f"The percentage of SLA violations with {model} checkpointing is: {SLA_percentages[model]:.2f}%")

ax = pd.Series(SLA_percentages).plot.bar(color="#2a78d6", edgecolor="white", rot=0)

# Percentage on top of each bar
for x, model in enumerate(checkpointing_models):
    ax.text(x, SLA_percentages[model], f"{SLA_percentages[model]:.1f}%", ha="center", va="bottom")

ax.set_ylim(top=max(SLA_percentages.values()) * 1.1)  # headroom for the labels
plt.title("Tasks delayed by more than 2 hours (SLA violations)")
plt.xlabel("Checkpointing model")
plt.ylabel("SLA violations (%)")
plt.show()
```

</details>

## Next steps

In this demo, we have shown how OpenDC can be used to simulate failures and to understand the impact of operational techniques that reduce their impact. We showed that using checkpointing can significantly improve the performance of a datacenter that is suffering from failures. However, using checkpointing also requires a good understanding of the trade-off between the overhead caused by making snapshots and the time saved by using them.

What else could you do with OpenDC regarding failures and checkpointing?
We suggest looking at:
1. What happens if we use a different failure model?
2. What happens when taking snapshots is more (or less) expensive?